# Skripsi: Integrasi Confident Learning dan CORN pada IndoBERT
**Notebook final** — Ryan Besto Saragih (NIM 23051204205) · Pembimbing: Anita Qoiriah, S.Kom., M.Kom.

Pipeline: scraping → kunci korpus (SHA-256) → praproses → split 70:10:20 grouped → ablasi proxy P1–P4
(+ kalibrasi/ECE, per-kelas, sensitivitas K) → deteksi noise → validasi manusia → 6 skenario × 3 seed →
uji signifikansi (**5 hipotesis pre-registered, H1–H5**) → uji emas → **sensitivitas split (3 partisi)**.

**Aturan penting**
1. Jalankan sel berurutan dari atas ke bawah. Sel scraping, split, proxy, dan training otomatis melewati hasil yang sudah ada.
2. **Jangan hapus** `all_reviews_master.csv`, `proxy_cache/`, `human_validation/`, `gold_test/`, `models_ckpt/`.
3. Split utama = seed 42. Validasi manusia dan uji emas **hanya** untuk split utama.
4. Sel bertanda ⚠️ BERHENTI memerlukan tindakan manual atau pengecekan sebelum lanjut.
5. Runtime Colab bisa putus: jalankan ulang dari Bagian 1, lalu lanjut ke sel yang terputus (auto-resume).

## Bagian 0 — Persiapan satu kali: salin folder Drive lama ke folder baru
Folder lama `SKRIPSI_CORN` **tidak diubah** (menjadi arsip). Sel ini aman dijalankan ulang: jika folder baru sudah berisi korpus, penyalinan dilewati.
Pastikan kode baru sudah di-push ke repo `corn-cl-indobert-rating-noise` sebelum menjalankan Bagian 1.

In [ ]:
from google.colab import drive
import os, subprocess
drive.mount('/content/drive')

OLD = "/content/drive/MyDrive/SKRIPSI_CORN"
NEW = "/content/drive/MyDrive/SKRIPSI_CORN_CL_FINAL"
MARK = os.path.join(NEW, "data", "raw", "all_reviews_master.csv")

if os.path.exists(MARK):
    print("✅ Folder baru sudah berisi korpus -- penyalinan dilewati.")
elif not os.path.exists(OLD):
    print("⛔ Folder lama tidak ditemukan:", OLD)
else:
    os.makedirs(NEW, exist_ok=True)
    print("⏳ Menyalin (bisa lama, termasuk checkpoint model)...")
    subprocess.run(["rsync", "-a", OLD + "/", NEW + "/"], check=True)
    print("✅ Selesai disalin.")

for name in ["data/raw", "data/processed", "proxy_cache", "cleaned", "models_ckpt",
             "human_validation", "gold_test", "results", "logs"]:
    p = os.path.join(NEW, name)
    n = sum(len(f) for _, _, f in os.walk(p)) if os.path.exists(p) else -1
    print(f"{name:<18} file: {n}")

## Bagian 0B — Persiapan run bersih (pembersihan terkontrol)
Folder lama `SKRIPSI_CORN` adalah **arsip dan tidak boleh disentuh**. Sel ini hanya membersihkan folder **FINAL**.

Pilih mode:
- **Mode A (disarankan)** — semua komputasi diulang dari data mentah, **kecuali** OOF proxy P4 dan anotasi manual (validasi manusia + uji emas). Alasan: OOF P4 menentukan baris mana yang dinilai manusia; menghitung ulangnya bisa mengubah baris ter-flag dan membuat 100 anotasi tidak cocok lagi. Nyatakan di Bab III bahwa OOF P4 berasal dari eksekusi pertama.
- **Mode B (benar-benar dari nol)** — OOF P4 juga dihitung ulang dan anotasi validasi manusia dibuat ulang (100 baris + 30 baris penilai kedua). Uji emas tetap dipertahankan jika split terbukti identik (sampelnya tidak bergantung pada proxy).

Dalam kedua mode, **tidak** dihapus: korpus mentah + manifest, `lexicon/`, `scrape_state/`, dan empat laporan hasil scraping (tidak bisa dibuat ulang tanpa scraping ulang).

`DRY_RUN = True` hanya menampilkan apa yang akan dihapus. Ubah ke `False` setelah daftarnya kamu baca.

In [ ]:
import os, shutil, hashlib

NEW = "/content/drive/MyDrive/SKRIPSI_CORN_CL_FINAL"
OLD = "/content/drive/MyDrive/SKRIPSI_CORN"
RUN_MODE = "A"      # "A" = pertahankan OOF P4 + anotasi ; "B" = dari nol (anotasi diulang)
DRY_RUN = True      # ubah ke False setelah daftar dicek

assert RUN_MODE in ("A", "B")
assert os.path.isdir(OLD), "Folder arsip lama tidak ditemukan -- JANGAN hapus apa pun sebelum arsip aman."
assert os.path.isdir(NEW), "Folder FINAL tidak ditemukan."

# 1) Korpus harus utuh (hash terkunci pada manifest asli)
EXPECTED_SHA = "c589b92cbe2fbc34d67e43a0a866a281cb31603a190af4517ef7f157e54589f6"
master = os.path.join(NEW, "data", "raw", "all_reviews_master.csv")
h = hashlib.sha256()
with open(master, "rb") as f:
    for chunk in iter(lambda: f.read(1 << 20), b""):
        h.update(chunk)
assert h.hexdigest() == EXPECTED_SHA, "SHA-256 korpus BERBEDA dari yang terkunci -- hentikan."
print("✅ Korpus utuh (SHA-256 cocok).")

# 2) Salinan manifest asli sebelum run_eda menulis ulang timestamp
res = os.path.join(NEW, "results")
m_src, m_bak = os.path.join(res, "dataset_manifest.json"), os.path.join(res, "dataset_manifest_locked_original.json")
if os.path.exists(m_src) and not os.path.exists(m_bak) and not DRY_RUN:
    shutil.copy2(m_src, m_bak)

KEEP_RESULTS = {"dataset_manifest.json", "dataset_manifest_locked_original.json", "scraping_summary.csv",
                "date_distribution_per_app_rating.csv", "text_length_summary.csv", "duplicate_review_texts.csv"}
KEEP_PROXY = ({"oof_pred_probs__finetuned_corn.npy", "oof_pred_probs_meta__finetuned_corn.csv"}
              if RUN_MODE == "A" else set())

to_delete = []
def add_children(sub, keep=frozenset()):
    p = os.path.join(NEW, sub)
    if os.path.isdir(p):
        for n in sorted(os.listdir(p)):
            if n not in keep:
                to_delete.append(os.path.join(p, n))

add_children("results", KEEP_RESULTS)
add_children("proxy_cache", KEEP_PROXY)
for sub in ["data/processed", "cleaned", "models_ckpt", "logs"]:
    add_children(sub)
if RUN_MODE == "B":
    add_children("human_validation")

def size_mb(p):
    if os.path.isfile(p): return os.path.getsize(p) / 1e6
    return sum(os.path.getsize(os.path.join(d, f)) for d, _, fs in os.walk(p) for f in fs) / 1e6

print(f"\nMODE {RUN_MODE} | DRY_RUN={DRY_RUN} | {len(to_delete)} item akan dihapus:")
for p in to_delete:
    print(f"  - {os.path.relpath(p, NEW):<70} {size_mb(p):>9.1f} MB")

print("\nDIPERTAHANKAN: data/raw, scrape_state, lexicon, gold_test",
      "| human_validation" if RUN_MODE == "A" else "| (human_validation DIHAPUS -> anotasi diulang)",
      "| proxy_cache/oof P4" if RUN_MODE == "A" else "")

if not DRY_RUN:
    for p in to_delete:
        shutil.rmtree(p) if os.path.isdir(p) else os.remove(p)
    print("\n🧹 Selesai. Folder FINAL siap untuk run bersih.")
else:
    print("\n(DRY_RUN) Belum ada yang dihapus. Set DRY_RUN=False lalu jalankan ulang sel ini.")

## Bagian 1 — Setup lingkungan
### Clone / pull repositori

In [ ]:
import os, sys

REPO_URL = "https://github.com/bestoism/corn-cl-indobert-rating-noise"
REPO_DIR = "/content/corn-cl-indobert-rating-noise"

if os.path.exists(REPO_DIR):
    print("🔄 Repo sudah ada, menarik update terbaru...")
    !cd {REPO_DIR} && git pull
else:
    print("⬇️  Clone repo baru...")
    !git clone {REPO_URL} {REPO_DIR}

sys.path.append(REPO_DIR)
os.chdir(REPO_DIR)
print(f"\n✅ Setup selesai. Working dir: {os.getcwd()}")

### Instalasi dependensi
`requirements.txt` di repo harus memuat: pandas, numpy, scikit-learn, torch, transformers, cleanlab, **coral-pytorch>=1.3.0**, statsmodels, scipy, matplotlib, tqdm, google-play-scraper, py3langid.

Catatan: `coral-pytorch` hanyalah **nama paket**; yang dipakai penelitian ini adalah fungsi **CORN** (`corn_loss`, `corn_label_from_logits`), bukan CORAL.

In [ ]:
!pip install -q -r requirements.txt
print("✅ Dependensi terpasang.")

### Import umum, verifikasi versi, dan catatan lingkungan
Sel ini menghentikan eksekusi bila `coral-pytorch` < 1.3.0 (versi lama memiliki normalisasi `corn_loss` yang berbeda dari paper).

In [ ]:
import os, platform
import pandas as pd
import numpy as np
import torch, transformers, cleanlab, coral_pytorch, sklearn, scipy, statsmodels
from packaging.version import Version

from src import config

assert Version(coral_pytorch.__version__) >= Version("1.3.0"), \
    "coral-pytorch < 1.3.0: normalisasi corn_loss berbeda dari paper! Upgrade dulu."

env = (f"python={platform.python_version()}\ntorch={torch.__version__}\n"
       f"transformers={transformers.__version__}\ncleanlab={cleanlab.__version__}\n"
       f"coral_pytorch={coral_pytorch.__version__}\nsklearn={sklearn.__version__}\n"
       f"scipy={scipy.__version__}\nstatsmodels={statsmodels.__version__}\n"
       f"gpu={torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'}\n")
print(env)
with open(os.path.join(config.PRIMARY_RESULTS_DIR, "environment.txt"), "w") as f:
    f.write(env)

print(f"📌 Proxy aktif: [{config.PROXY_ID}] {config.PROXY_NAME}")
print(f"🧩 Split aktif: seed={config.SPLIT_SEED}")
print(f"📁 Drive root : {config.DRIVE_ROOT}")
assert "SKRIPSI_CORN_CL_FINAL" in config.DRIVE_ROOT, "Drive root bukan folder FINAL -- cek config.py"

## Bagian 2 — Scraping Google Play Store
Dijalankan **sekali seumur proyek**. Otomatis dilewati karena `all_reviews_master.csv` sudah ada. Jangan hapus file itu kecuali sengaja scraping ulang dari nol (semua eksperimen berikutnya harus diulang).

In [ ]:
from src import config

if os.path.exists(config.RAW_DATA_FILE):
    print(f"✅ {config.RAW_DATA_FILE} sudah ada -- scraping dilewati.")
    df_check = pd.read_csv(config.RAW_DATA_FILE)
    print(f"   Baris tersedia: {len(df_check)}")
else:
    from scripts.scrape_google_play import main as run_scraping
    run_scraping()

## Bagian 3 — Kunci korpus + EDA awal (sebelum praproses)
Dedup `review_id`, manifest + SHA-256, dan seluruh angka deskripsi data Subbab 3.2.

In [ ]:
# Pagar integritas: hash dibandingkan dengan nilai terkunci (bukan dengan manifest yang akan ditulis ulang run_eda)
import hashlib
EXPECTED_SHA = "c589b92cbe2fbc34d67e43a0a866a281cb31603a190af4517ef7f157e54589f6"
h = hashlib.sha256()
with open(config.RAW_DATA_FILE, "rb") as f:
    for chunk in iter(lambda: f.read(1 << 20), b""):
        h.update(chunk)
assert h.hexdigest() == EXPECTED_SHA, "Korpus BERBEDA dari yang terkunci -- hentikan semua eksperimen."
print("✅ SHA-256 korpus cocok dengan nilai terkunci.")

In [ ]:
from scripts.eda_dataset import main as run_eda
run_eda()

### Verifikasi korpus terhadap manifest (SHA-256)
Hasil harus **COCOK**. Jika tidak, korpus berubah dan seluruh eksperimen harus diulang — hentikan di sini.

In [ ]:
from scripts.eda_dataset import verify_manifest
verify_manifest()
# Hash yang diharapkan: c589b92cbe2fbc34d67e43a0a866a281cb31603a190af4517ef7f157e54589f6

## Bagian 4 — Praproses teks + split 70:10:20 (grouped, stratified, seed 42)
Dijalankan sekali; dilewati bila ketiga file split sudah ada. Split dikunci dan dipakai seluruh eksperimen split utama.

In [ ]:
from src.preprocess import run_preprocessing
from src.data_split import create_splits
from src import config

config.set_split(42)

if (os.path.exists(config.TRAIN_RAW_FILE) and os.path.exists(config.VAL_FILE)
        and os.path.exists(config.TEST_FILE)):
    print("✅ Split train/val/test sudah ada -- dilewati.")
    df_train = pd.read_csv(config.TRAIN_RAW_FILE)
    df_val = pd.read_csv(config.VAL_FILE)
    df_test = pd.read_csv(config.TEST_FILE)
    print(f"   Train: {len(df_train)} | Val: {len(df_val)} | Test: {len(df_test)}")
else:
    print("=" * 60); print(" PREPROCESSING "); print("=" * 60)
    df_clean = run_preprocessing(config.RAW_DATA_FILE, config.CLEAN_TEXT_FILE)
    df_clean = df_clean.dropna(subset=["cleaned_text", "rating"])
    print("\n" + "=" * 60)
    print(" SPLIT 70:10:20 (grouped teks, stratified rating, seed=42) ")
    print("=" * 60)
    df_train, df_val, df_test = create_splits(config.CLEAN_TEXT_FILE, seed=42)

### Bukti tidak ada kebocoran teks lintas split
Ketiga angka harus **0**. Bandingkan dengan simulasi split acak non-grouped di EDA (16,58% teks uji muncul di train) — ini membenarkan grouped split (Subbab 3.4).

In [ ]:
tr, va, te = [pd.read_csv(p) for p in (config.TRAIN_RAW_FILE, config.VAL_FILE, config.TEST_FILE)]
T = lambda d: set(d["cleaned_text"])
print("overlap train∩val, train∩test, val∩test:",
      len(T(tr) & T(va)), len(T(tr) & T(te)), len(T(va) & T(te)))

### ⚠️ Cek: split baru identik dengan split arsip?
Uji emas (92 baris) dan OOF P4 hanya valid bila split train/val/test **identik** dengan run pertama. Semua baris harus `True`. Jika ada `False`, jangan pakai ulang anotasi dan cache lama (pakai Mode B dan buat sampel uji emas baru).

In [ ]:
OLD_SPLIT = "/content/drive/MyDrive/SKRIPSI_CORN/data/processed"
all_same = True
for nm, new_path in [("split_train_raw.csv", config.TRAIN_RAW_FILE), ("split_val.csv", config.VAL_FILE),
                     ("split_test.csv", config.TEST_FILE)]:
    a = pd.read_csv(os.path.join(OLD_SPLIT, nm)); b = pd.read_csv(new_path)
    same_ids = set(a["review_id"]) == set(b["review_id"])
    same_order_text = a["cleaned_text"].tolist() == b["cleaned_text"].tolist()
    all_same &= same_ids and same_order_text
    print(f"{nm:<22} baris lama={len(a)} baru={len(b)} | review_id sama={same_ids} | urutan+teks sama={same_order_text}")
print("\nSplit identik dengan arsip:", "YA ✅" if all_same else "TIDAK ❌ -- anotasi lama & cache OOF lama TIDAK valid")

### Sampling subset uji emas (Subbab 3.10) + rubrik anotasi
Anotasi bisa dikerjakan paralel sambil menunggu training. **Hanya split utama.** Jika file sudah ada, tidak ditimpa.
Klaim "rubrik disediakan" di laporan hanya boleh dipertahankan jika anotasi memang memakai rubrik ini.

In [ ]:
from src import config, gold_test

config.set_split(42)
if os.path.exists(config.GOLD_TEST_SAMPLE_FILE):
    print(f"✅ {config.GOLD_TEST_SAMPLE_FILE} sudah ada -- sampling dilewati.")
else:
    gold_test.sample_gold_test_subset(seed=42)
    gold_test.export_second_annotator_gold_subset()
gold_test.write_rubric()

## Bagian 5 — Ablasi proxy classifier P1–P4 (Subbab 3.5, Tabel 3.1)
P3/P4 memakai **cache OOF yang sudah ada** (tidak dilatih ulang), P1/P2 memakai cache embedding. Sel ini menghasilkan file diagnostik baru: kalibrasi/ECE, kualitas per kelas, distribusi noise per rating.

⚠️ Jika cache tidak terbaca (misalnya path Drive salah), kode akan **melatih ulang** P3/P4 dan hasilnya bisa berbeda dari anotasi manual. Sel uji regresi di bawahnya wajib dicek.

In [ ]:
from src import config
from src.clean import run_confident_learning

config.set_split(42)
for pid in [0, 1, 2, 3]:
    print(f"\n{'='*70}\n ABLASI PROXY -- PROXY_ID = {pid} \n{'='*70}")
    config.set_proxy(pid)
    try:
        run_confident_learning()
    except Exception as e:
        print(f"⚠️ Proxy {pid} gagal: {e}")

config.set_proxy(3)   # kunci kembali ke proxy final P4
print("\n✅ Ablasi P1–P4 selesai.")
display(pd.read_csv(config.PROXY_QUALITY_LOG_FILE))

### ⚠️ Pemeriksaan hasil ablasi
- **Mode A:** P4 **harus persis** 4732 baris ter-flag (dimuat dari cache). Jika berbeda, **berhenti**: cache tidak terbaca dan anotasi tidak lagi cocok.
- **Mode B:** semua angka baru adalah angka resmi; angka run pertama hanya pembanding dan tidak dipakai lagi.
- P1–P3 boleh berbeda sedikit dari run pertama (GPU/paralelisme tidak deterministik penuh); itu normal.

In [ ]:
RUN_MODE = "A"   # samakan dengan sel Bagian 0B
PILOT = {0: 4578, 1: 4600, 2: 4857, 3: 4732}   # run pertama (confident_learning)
tbl = pd.read_csv(config.PROXY_QUALITY_LOG_FILE).set_index("proxy_id")
for pid, exp in PILOT.items():
    got = int(tbl.loc[pid, "n_issues_confident_learning"])
    print(f"P{pid+1} {config.PROXY_REGISTRY[pid]['name']:<20} run pertama={exp}  run ini={got}")
if RUN_MODE == "A":
    assert int(tbl.loc[3, "n_issues_confident_learning"]) == PILOT[3], \
        "P4 BEDA dari run pertama -- cache OOF tidak terbaca; JANGAN lanjut (anotasi tidak lagi cocok)."
    print("\n✅ P4 identik dengan run pertama (cache terbaca). Anotasi manual tetap valid.")
else:
    print("\nMode B: angka baru ini yang dipakai di laporan; anotasi validasi manusia harus diisi ulang.")

## Bagian 6 — Kunci proxy final (P4) + sampel validasi manusia
Menjalankan Confident Learning dengan P4 (dari cache). File sampel validasi manusia **tidak akan ditimpa** bila sudah ada (guard di `clean.py`); sel ini harus mencetak pesan `🛑 ... TIDAK ditimpa`.

In [ ]:
from src import config
from src.clean import run_confident_learning

config.set_split(42)
config.set_proxy(3)
print(f"🔒 Proxy final terkunci: [{config.PROXY_ID}] {config.PROXY_NAME}")
df_noise, proxy_metrics = run_confident_learning()

## Bagian 7 — Validasi manusia (Subbab 3.7)
Anotasi 100 baris **sudah selesai** pada run pilot (noise 26%, not_noise 60%, ambiguous 14%). Sel di bawah hanya membaca ulang dan menghitung agreement. Jangan mengisi ulang file anotasi.

In [ ]:
df_hv = pd.read_csv(config.HUMAN_VALIDATION_FILE, sep=None, engine="python")
print(df_hv.shape)
print(df_hv["human_verdict"].value_counts(dropna=False))

In [ ]:
from src.human_validation import compute_agreement
result = compute_agreement()
print("\n✅ Validasi manusia lengkap." if result is not None else "\n⛔ Belum lengkap/ada error -- perbaiki dulu.")

### Anotator kedua / Cohen's kappa
Jika file anotator kedua sudah terisi (kappa run pilot = 0,3797, n=30), cukup hitung ulang. Jalur fallback single-annotator tersedia di komentar.

In [ ]:
from src import human_validation as hv

if not os.path.exists(config.HUMAN_VALIDATION_ANNOTATOR2_FILE):
    hv.export_second_annotator_subset()
    print("⚠️ Minta penilai kedua mengisi 'human_verdict' (BUTA), lalu jalankan sel berikutnya.")
else:
    print("✅ File anotator kedua sudah ada.")

In [ ]:
kappa = hv.compute_interannotator_kappa()          # jika ada penilai kedua
# Fallback satu penilai:
# hv.export_retest_subset(); (isi manual) ; kappa = hv.compute_test_retest_reliability()

## Bagian 8 — Training 6 skenario × 3 seed + uji signifikansi (split utama, seed 42)
Idempoten: skenario/seed yang sudah selesai dilewati. Pada split utama, checkpoint dan `experiment_progress.json` dari run pilot dipakai ulang sehingga **tidak ada training ulang**.

Selisih QWK antar-fungsi loss bergantung cara agregasi, maka dilaporkan dua versi berdampingan:
- `qwk_effect_sizes.csv` — agregasi **ensemble** 3 seed (definisi utama, Subbab 3.12)
- `qwk_per_seed_effect_sizes.csv` — rata-rata QWK **per seed** (analisis sensitivitas)

### ⚠️ Pemeriksaan sebelum training (mencegah training ulang tidak sengaja)

In [ ]:
import glob, json
from src import config

config.set_split(42); config.set_proxy(3)
ckpts = glob.glob(os.path.join(config.MODEL_CKPT_DIR, "*", "seed*_best.pt"))
print("Checkpoint ditemukan:", len(ckpts), "(diharapkan 18)")
print("Progress file ada   :", os.path.exists(config.PROGRESS_FILE), "->", config.PROGRESS_FILE)

if len(ckpts) == 18 and not os.path.exists(config.PROGRESS_FILE):
    raise RuntimeError("Checkpoint lengkap tapi progress file hilang -- training akan diulang! "
                       "Cek folder logs/ di Drive sebelum lanjut.")
if os.path.exists(config.PROGRESS_FILE):
    prog = json.load(open(config.PROGRESS_FILE))
    print("Skenario di progress:", {k: len(v) for k, v in prog.items()})

In [ ]:
from src import config
from src.split_sensitivity import run_split_pipeline, build_scenarios, make_final_table

config.set_split(42); config.set_proxy(3)
run_split_pipeline(42)
scenarios = build_scenarios()      # dipakai sel uji emas

print("\n🏆 HASIL 6 SKENARIO (split 42, mean ± std 3 seed):")
display(make_final_table())

In [ ]:
R = config.RESULTS_DIR
for f in ["significance_test.csv", "effect_sizes.csv", "qwk_effect_sizes.csv", "qwk_per_seed_effect_sizes.csv"]:
    print(f"\n=== {f} ===")
    display(pd.read_csv(os.path.join(R, f)))
print("\nCatatan: laporkan QWK ensemble (definisi utama) dan QWK per-seed berdampingan dengan penjelasan perbedaannya.")

## Bagian 9 — Evaluasi pada subset uji emas (Subbab 3.10)
Pemeriksaan ketahanan **sekunder**; tidak dipakai untuk uji formal H1–H5. **Hanya split utama.** Pastikan kolom `human_gold_rating` sudah terisi lengkap (1–5 atau `ND`).

In [ ]:
from src import config, gold_test
config.set_split(42); config.set_proxy(3)

result = gold_test.evaluate_on_gold_subset(scenarios)
if result is not None:
    df_gold_result, df_gold_effect = result
    print("\n📌 Bandingkan arah di atas dengan H1–H5 pada data uji penuh: konsisten = memperkuat validitas; "
          "tidak konsisten = dilaporkan sebagai temuan yang membatasi generalisasi (Subbab 3.10).")
    gold_test.compute_gold_interannotator_kappa()
else:
    print(f"\n⛔ Lengkapi dulu {config.GOLD_TEST_SAMPLE_FILE} sebelum menjalankan sel ini.")

## Bagian 10 — Sensitivitas jumlah fold K pada proxy P4 (Subbab 3.5)
K = 5 memakai cache yang ada. K = 3 dan K = 10 melatih proxy baru (**±130 menit GPU**). Bersifat konfirmatif terhadap keputusan apriori K = 5, bukan pencarian nilai optimum. Hanya split utama.

In [ ]:
from src.k_sensitivity import run_k_sensitivity
config.set_split(42); config.set_proxy(3)
display(run_k_sensitivity())

## Bagian 11 — Sensitivitas partisi data (3 split acak: 42, 123, 2024)
Menjawab keberatan pembimbing bahwa satu split kurang representatif. Aturan:
- Split 42 = utama (sudah selesai di Bagian 8). Split 123 dan 2024 = sensitivitas.
- **Semua dilaporkan; tidak ada split yang "dipilih"** berdasarkan hasil.
- Seed partisi (42/123/2024) berbeda dari seed bobot (42/123/2024).
- Tidak ada p-value gabungan antar-split (test set tumpang tindih, bukan replikasi independen).
- Aturan konsistensi (ditulis di muka): konsisten bila arah sama dan CI 95% tidak melewati nol pada ≥ 2 dari 3 split.

Waktu per split: ±9–13 jam GPU (3 seed bobot) atau ±3–4 jam (opsi hemat `seed_list=[42]`).
**Jalankan satu split per sesi Colab**; bila runtime putus, jalankan ulang sel yang sama (auto-resume).

In [ ]:
from src.split_sensitivity import run_split_pipeline

MODE_HEMAT = False     # True = 1 seed bobot per split (dinyatakan sebagai keterbatasan di laporan)
SEEDS = [42] if MODE_HEMAT else None

run_split_pipeline(123, seed_list=SEEDS)

In [ ]:
run_split_pipeline(2024, seed_list=SEEDS)

In [ ]:
from src.split_sensitivity import summarize_across_splits

long, summ = summarize_across_splits()      # otomatis kembali ke split 42
print("=== Rinci per split ===");  display(long)
print("\n=== Ringkasan konsistensi lintas split ===");  display(summ)

## Bagian 12 — Ringkasan akhir (siap disalin ke Bab IV)
Dibaca dari file di Drive, jadi aman dijalankan kapan saja.

In [ ]:
import os
import pandas as pd
from src import config
config.set_split(42); config.set_proxy(3)
P = config.PRIMARY_RESULTS_DIR

def show(title, path):
    print(f"\n{title}")
    if os.path.exists(path): display(pd.read_csv(path))
    else: print("   ⚠️ belum ada:", path)

print("=" * 70); print(" RINGKASAN LENGKAP (split utama, seed 42) "); print("=" * 70)
show("[1] TABEL ABLASI PROXY (P1-P4)", config.PROXY_QUALITY_LOG_FILE)

print("\n[2] LAPORAN DATA & EDA (Bab III):")
for f in ["scraping_summary.csv", "date_distribution_per_app_rating.csv", "text_length_summary.csv",
          "duplicate_review_texts.csv", "dataset_manifest.json", "split_summary.csv",
          "preprocessing_summary.csv", "token_truncation_summary.csv"]:
    print("  ", "✅" if os.path.exists(os.path.join(P, f)) else "⚠️ belum ada", f)

print("\n[3] DIAGNOSTIK PROXY (per kelas, ECE, distribusi noise per rating, suhu per fold):")
for f in sorted(os.listdir(P)):
    if f.startswith(("calibration_report__", "proxy_per_class__", "noise_distribution_by_rating__", "temperature_per_fold__")):
        print("   ✅", f)

show("[4] HASIL 6 SKENARIO (mean ± std)", config.FINAL_RESULTS_TABLE_FILE)
show("[5] UJI SIGNIFIKANSI H1-H5", config.SIGNIFICANCE_TEST_FILE)
show("[6] EFFECT SIZE MAE + CI 95%", os.path.join(P, "effect_sizes.csv"))
show("[7a] EFFECT SIZE QWK (ensemble, definisi utama)", os.path.join(P, "qwk_effect_sizes.csv"))
show("[7b] EFFECT SIZE QWK (per-seed, sensitivitas)", os.path.join(P, "qwk_per_seed_effect_sizes.csv"))
show("[8] UJI EMAS", config.GOLD_TEST_EVAL_FILE)
show("[9] ARAH & EFFECT SIZE UJI EMAS", config.GOLD_TEST_EFFECT_SIZE_FILE)
show("[10] SENSITIVITAS K", config.K_SENSITIVITY_FILE)
show("[11] SENSITIVITAS SPLIT (ringkasan)", os.path.join(P, "split_sensitivity_summary.csv"))

print("\n[12] VALIDASI MANUSIA:")
if os.path.exists(config.HUMAN_VALIDATION_RESULT_FILE):
    d = pd.read_csv(config.HUMAN_VALIDATION_RESULT_FILE); c = d["human_verdict"].value_counts()
    print(f"   Total {len(d)} | noise {c.get('noise',0)} | not_noise {c.get('not_noise',0)} | ambiguous {c.get('ambiguous',0)}")
show("[13] KAPPA ANTAR-ANOTATOR", config.HUMAN_VALIDATION_KAPPA_FILE)

env_path = os.path.join(P, "environment.txt")
print("\n[14] LINGKUNGAN:\n", open(env_path).read() if os.path.exists(env_path) else "belum ada")
print("✅ Semua file hasil di:", P)